In [1]:
import re
import pdfplumber
from typing import Dict, List, Optional, Set, Any, Tuple
from dataclasses import dataclass, field
import chromadb

@dataclass
class MonsterAbilityScores:
    strength: int
    dexterity: int
    constitution: int
    intelligence: int
    wisdom: int
    charisma: int

@dataclass
class MonsterAttack:
    name: str
    attack_bonus: int
    damage_dice: str
    damage_type: str
    range: Optional[str] = None
    description: Optional[str] = None

@dataclass
class MonsterCombatStats:
    armor_class: int
    hit_points: int
    hit_dice: str
    speed: Dict[str, str]
    ability_scores: MonsterAbilityScores
    saving_throws: Optional[Dict[str, int]] = None
    skills: Optional[Dict[str, int]] = None

@dataclass
class Monster:
    name: str
    size: str
    creature_type: str
    alignment: str
    challenge_rating: str
    combat_stats: MonsterCombatStats
    attacks: List[MonsterAttack]
    special_abilities: List[str]
    description: str
    environment: List[str]
    senses: Optional[str] = None
    languages: Optional[str] = None
    damage_resistances: Optional[List[str]] = None
    damage_immunities: Optional[List[str]] = None
    condition_immunities: Optional[List[str]] = None
    traits: Optional[List[str]] = None
    actions: Optional[List[str]] = None
    reactions: Optional[List[str]] = None
    legendary_actions: Optional[List[str]] = None

class ImprovedMonsterParser:
    """Enhanced Monster Manual PDF Parser that starts at Aboleth"""

    def __init__(self):
        # Known monster names to validate parsing start
        self.first_monsters = ["ABOLETH", "ANGELS", "DEVA", "PLANETAR", "SOLAR"]

        # Improved regex patterns
        self.patterns = {
            'monster_header': r'^([A-Z][A-Z\s\'-]+)$',
            'size_type_alignment': r'(Tiny|Small|Medium|Large|Huge|Gargantuan)\s+(aberration|beast|celestial|construct|dragon|elemental|fey|fiend|giant|humanoid|monstrosity|ooze|plant|undead),?\s+(.+?)(?:\n|$)',
            'armor_class': r'Armor Class\s+(\d+)(?:\s*\(([^)]+)\))?',
            'hit_points': r'Hit Points\s+(\d+)\s*\(([^)]+)\)',
            'speed': r'Speed\s+(.+?)(?=\n[A-Z]|\n\n|\Z)',
            'ability_scores': r'STR\s+DEX\s+CON\s+INT\s+WIS\s+CHA\s+(\d+)\s+\([+-]?\d+\)\s+(\d+)\s+\([+-]?\d+\)\s+(\d+)\s+\([+-]?\d+\)\s+(\d+)\s+\([+-]?\d+\)\s+(\d+)\s+\([+-]?\d+\)\s+(\d+)\s+\([+-]?\d+\)',
            'saving_throws': r'Saving Throws\s+(.+?)(?=\n[A-Z]|\n\n|\Z)',
            'skills': r'Skills\s+(.+?)(?=\n[A-Z]|\n\n|\Z)',
            'damage_resistances': r'Damage Resistances\s+(.+?)(?=\n[A-Z]|\n\n|\Z)',
            'damage_immunities': r'Damage Immunities\s+(.+?)(?=\n[A-Z]|\n\n|\Z)',
            'condition_immunities': r'Condition Immunities\s+(.+?)(?=\n[A-Z]|\n\n|\Z)',
            'senses': r'Senses\s+(.+?)(?=\n[A-Z]|\n\n|\Z)',
            'languages': r'Languages\s+(.+?)(?=\n[A-Z]|\n\n|\Z)',
            'challenge': r'Challenge\s+([^\s\(]+)',
            'actions_section': r'Actions\s*(.+?)(?=Legendary Actions|Reactions|\n[A-Z][A-Z\s\'-]+\n|\Z)',
            'legendary_actions_section': r'Legendary Actions\s*(.+?)(?=Reactions|\n[A-Z][A-Z\s\'-]+\n|\Z)',
            'reactions_section': r'Reactions\s*(.+?)(?=Legendary Actions|\n[A-Z][A-Z\s\'-]+\n|\Z)'
        }

    def extract_pdf_text(self, pdf_path: str) -> Optional[str]:
        """Extract text from Monster Manual PDF with better error handling"""
        try:
            print(f"Extracting text from {pdf_path}...")
            full_text = ""

            with pdfplumber.open(pdf_path) as pdf:
                total_pages = len(pdf.pages)
                print(f"Processing {total_pages} pages...")

                for page_num, page in enumerate(pdf.pages, 1):
                    if page_num % 50 == 0:
                        print(f"Processed {page_num}/{total_pages} pages...")

                    try:
                        page_text = page.extract_text()
                        if page_text:
                            full_text += page_text + "\n\n"
                    except Exception as e:
                        print(f"Warning: Could not extract page {page_num}: {e}")
                        continue

            print(f"Extraction complete! {len(full_text):,} characters extracted")
            return full_text

        except Exception as e:
            print(f"PDF extraction failed: {e}")
            return None

    def find_aboleth_start(self, text: str) -> int:
        """Find where Aboleth appears to start parsing from there"""
        lines = text.split('\n')

        for i, line in enumerate(lines):
            line_clean = line.strip().upper()
            if line_clean == "ABOLETH" and self._looks_like_monster_header(line_clean):
                print(f"Found Aboleth at line {i}")
                return i

        # Fallback: look for any of the first monsters
        for i, line in enumerate(lines):
            line_clean = line.strip().upper()
            if line_clean in self.first_monsters:
                print(f"Found {line_clean} at line {i} as starting point")
                return i

        print("Warning: Could not find Aboleth or other starting monster. Starting from beginning.")
        return 0

    def _looks_like_monster_header(self, line: str) -> bool:
        """Check if a line looks like a monster header"""
        return (line.isupper() and
                2 <= len(line) <= 50 and
                not any(word in line for word in ['CONTENTS', 'CHAPTER', 'INDEX', 'CREDITS', 'PAGE']))

    def split_into_monster_blocks(self, text: str) -> List[str]:
        """Split text into individual monster stat blocks starting from Aboleth"""
        lines = text.split('\n')

        # Find starting point
        start_line = self.find_aboleth_start(text)
        lines = lines[start_line:]

        monster_blocks = []
        current_block = []

        for line in lines:
            line = line.strip()

            # Skip empty lines
            if not line:
                continue

            # Check if this starts a new monster
            if self._looks_like_monster_header(line):
                # Save previous block if it exists and has substantial content
                if current_block and len('\n'.join(current_block)) > 200:
                    monster_blocks.append('\n'.join(current_block))
                current_block = [line]
            else:
                current_block.append(line)

        # Don't forget the last block
        if current_block and len('\n'.join(current_block)) > 200:
            monster_blocks.append('\n'.join(current_block))

        print(f"Split into {len(monster_blocks)} monster blocks")
        return monster_blocks

    def parse_monster_block(self, block_text: str) -> Optional[Monster]:
        """Parse a single monster stat block with improved error handling"""
        try:
            # Basic validation
            if len(block_text) < 100:
                return None

            lines = [line.strip() for line in block_text.split('\n') if line.strip()]

            # Extract monster name (should be first line)
            name = self._extract_name(lines)
            if not name:
                return None

            # Extract basic info
            size, creature_type, alignment = self._extract_size_type_alignment(block_text)
            if not size or not creature_type:
                return None

            # Extract combat stats
            armor_class = self._extract_armor_class(block_text)
            hit_points, hit_dice = self._extract_hit_points(block_text)
            speed = self._extract_speed(block_text)
            abilities = self._extract_ability_scores(block_text)

            # Extract challenge rating
            challenge_rating = self._extract_challenge_rating(block_text)

            # Extract optional stats
            saving_throws = self._extract_saving_throws(block_text)
            skills = self._extract_skills(block_text)
            senses = self._extract_senses(block_text)
            languages = self._extract_languages(block_text)
            damage_resistances = self._extract_damage_resistances(block_text)
            damage_immunities = self._extract_damage_immunities(block_text)
            condition_immunities = self._extract_condition_immunities(block_text)

            # Extract abilities and actions
            attacks = self._extract_attacks(block_text)
            special_abilities = self._extract_special_abilities(block_text)
            traits = self._extract_traits(block_text)
            legendary_actions = self._extract_legendary_actions(block_text)

            # Create combat stats object
            combat_stats = MonsterCombatStats(
                armor_class=armor_class or 10,
                hit_points=hit_points or 1,
                hit_dice=hit_dice or "1d8",
                speed=speed or {"walk": "30 ft"},
                ability_scores=abilities or MonsterAbilityScores(10, 10, 10, 10, 10, 10),
                saving_throws=saving_throws,
                skills=skills
            )

            # Extract description and environment
            description = self._extract_description(block_text)
            environment = self._infer_environment(name, block_text)

            return Monster(
                name=name,
                size=size,
                creature_type=creature_type,
                alignment=alignment or "Unaligned",
                challenge_rating=challenge_rating or "1",
                combat_stats=combat_stats,
                attacks=attacks,
                special_abilities=special_abilities,
                description=description,
                environment=environment,
                senses=senses,
                languages=languages,
                damage_resistances=damage_resistances,
                damage_immunities=damage_immunities,
                condition_immunities=condition_immunities,
                traits=traits,
                legendary_actions=legendary_actions
            )

        except Exception as e:
            print(f"Error parsing monster block: {e}")
            return None

    def _extract_name(self, lines: List[str]) -> Optional[str]:
        """Extract monster name from first line"""
        if not lines:
            return None

        first_line = lines[0].strip()
        if self._looks_like_monster_header(first_line):
            return first_line.title()

        return None

    def _extract_size_type_alignment(self, text: str) -> Tuple[Optional[str], Optional[str], Optional[str]]:
        """Extract size, creature type, and alignment"""
        match = re.search(self.patterns['size_type_alignment'], text, re.IGNORECASE | re.MULTILINE)
        if match:
            size = match.group(1)
            creature_type = match.group(2)
            alignment = match.group(3).strip()
            # Clean up alignment
            alignment = re.sub(r'[^\w\s]', '', alignment).strip()
            return size, creature_type, alignment
        return None, None, None

    def _extract_armor_class(self, text: str) -> Optional[int]:
        """Extract armor class"""
        match = re.search(self.patterns['armor_class'], text, re.IGNORECASE)
        if match:
            try:
                return int(match.group(1))
            except ValueError:
                pass
        return None

    def _extract_hit_points(self, text: str) -> Tuple[Optional[int], Optional[str]]:
        """Extract hit points and hit dice"""
        match = re.search(self.patterns['hit_points'], text, re.IGNORECASE)
        if match:
            try:
                hp = int(match.group(1))
                hit_dice = match.group(2)
                return hp, hit_dice
            except ValueError:
                pass
        return None, None

    def _extract_speed(self, text: str) -> Optional[Dict[str, str]]:
        """Extract speed information"""
        match = re.search(self.patterns['speed'], text, re.IGNORECASE | re.DOTALL)
        if not match:
            return None

        speed_text = match.group(1).strip()
        speed_dict = {}

        # Parse different movement types
        movement_patterns = {
            'walk': r'(?:^|\s)(\d+)\s*ft\.?(?:\s|,|$)',
            'fly': r'fly\s+(\d+)\s*ft\.?',
            'swim': r'swim\s+(\d+)\s*ft\.?',
            'burrow': r'burrow\s+(\d+)\s*ft\.?',
            'climb': r'climb\s+(\d+)\s*ft\.?'
        }

        for move_type, pattern in movement_patterns.items():
            match = re.search(pattern, speed_text, re.IGNORECASE)
            if match:
                speed_dict[move_type] = f"{match.group(1)} ft"

        return speed_dict if speed_dict else {"walk": "30 ft"}

    def _extract_ability_scores(self, text: str) -> Optional[MonsterAbilityScores]:
        """Extract ability scores"""
        match = re.search(self.patterns['ability_scores'], text, re.MULTILINE)
        if match:
            try:
                return MonsterAbilityScores(
                    strength=int(match.group(1)),
                    dexterity=int(match.group(2)),
                    constitution=int(match.group(3)),
                    intelligence=int(match.group(4)),
                    wisdom=int(match.group(5)),
                    charisma=int(match.group(6))
                )
            except (ValueError, IndexError):
                pass

        return MonsterAbilityScores(10, 10, 10, 10, 10, 10)

    def _extract_challenge_rating(self, text: str) -> Optional[str]:
        """Extract challenge rating"""
        match = re.search(self.patterns['challenge'], text, re.IGNORECASE)
        if match:
            return match.group(1).strip()
        return None

    def _extract_saving_throws(self, text: str) -> Optional[Dict[str, int]]:
        """Extract saving throws"""
        match = re.search(self.patterns['saving_throws'], text, re.IGNORECASE)
        if not match:
            return None

        saves_text = match.group(1)
        saves_dict = {}

        # Parse saves like "Str +5, Dex +3"
        save_matches = re.findall(r'(\w+)\s*([+-]?\d+)', saves_text)
        for ability, bonus in save_matches:
            ability_name = self._standardize_ability_name(ability)
            if ability_name:
                saves_dict[ability_name] = int(bonus)

        return saves_dict if saves_dict else None

    def _extract_skills(self, text: str) -> Optional[Dict[str, int]]:
        """Extract skills"""
        match = re.search(self.patterns['skills'], text, re.IGNORECASE)
        if not match:
            return None

        skills_text = match.group(1)
        skills_dict = {}

        # Parse skills like "Perception +5, Stealth +3"
        skill_matches = re.findall(r'([A-Za-z\s]+)\s*([+-]?\d+)', skills_text)
        for skill, bonus in skill_matches:
            skill_clean = skill.strip().title()
            try:
                skills_dict[skill_clean] = int(bonus)
            except ValueError:
                continue

        return skills_dict if skills_dict else None

    def _extract_senses(self, text: str) -> Optional[str]:
        """Extract senses"""
        match = re.search(self.patterns['senses'], text, re.IGNORECASE)
        return match.group(1).strip() if match else None

    def _extract_languages(self, text: str) -> Optional[str]:
        """Extract languages"""
        match = re.search(self.patterns['languages'], text, re.IGNORECASE)
        return match.group(1).strip() if match else None

    def _extract_damage_resistances(self, text: str) -> Optional[List[str]]:
        """Extract damage resistances"""
        match = re.search(self.patterns['damage_resistances'], text, re.IGNORECASE)
        if match:
            return [r.strip() for r in match.group(1).split(',')]
        return None

    def _extract_damage_immunities(self, text: str) -> Optional[List[str]]:
        """Extract damage immunities"""
        match = re.search(self.patterns['damage_immunities'], text, re.IGNORECASE)
        if match:
            return [i.strip() for i in match.group(1).split(',')]
        return None

    def _extract_condition_immunities(self, text: str) -> Optional[List[str]]:
        """Extract condition immunities"""
        match = re.search(self.patterns['condition_immunities'], text, re.IGNORECASE)
        if match:
            return [c.strip() for c in match.group(1).split(',')]
        return None

    def _extract_attacks(self, text: str) -> List[MonsterAttack]:
        """Extract attacks from Actions section"""
        attacks = []

        # Find Actions section
        actions_match = re.search(self.patterns['actions_section'], text, re.IGNORECASE | re.DOTALL)
        if not actions_match:
            return attacks

        actions_text = actions_match.group(1)

        # Look for attack patterns
        attack_patterns = [
            r'([A-Za-z\s]+)\.\s*[^.]*?[+–-](\d+)\s+to hit[^.]*?(\d+d\d+(?:\s*[+-]\s*\d+)?)\s+(\w+)\s+damage',
            r'([A-Za-z\s]+)\s+Attack[^.]*?[+–-](\d+)\s+to hit[^.]*?(\d+d\d+(?:\s*[+-]\s*\d+)?)\s+(\w+)\s+damage'
        ]

        for pattern in attack_patterns:
            matches = re.findall(pattern, actions_text, re.IGNORECASE)
            for match in matches:
                try:
                    attack = MonsterAttack(
                        name=match[0].strip(),
                        attack_bonus=int(match[1]),
                        damage_dice=match[2].strip(),
                        damage_type=match[3].strip()
                    )
                    attacks.append(attack)
                except (ValueError, IndexError):
                    continue

        return attacks

    def _extract_special_abilities(self, text: str) -> List[str]:
        """Extract special abilities/traits before Actions section"""
        abilities = []

        # Split text into sections and look for traits before Actions
        sections = re.split(r'\n(Actions|Legendary Actions|Reactions)\n', text, flags=re.IGNORECASE)
        if len(sections) > 0:
            traits_section = sections[0]

            # Look for ability patterns
            ability_matches = re.findall(r'([A-Z][^.\n]+?)\.\s*([^.\n]+(?:\.[^.\n]+)*)', traits_section)
            for match in ability_matches:
                if len(match[0]) < 50:  # Reasonable ability name length
                    abilities.append(f"{match[0].strip()}: {match[1].strip()}")

        return abilities

    def _extract_traits(self, text: str) -> List[str]:
        """Extract traits (same as special abilities for now)"""
        return self._extract_special_abilities(text)

    def _extract_legendary_actions(self, text: str) -> List[str]:
        """Extract legendary actions"""
        legendary_match = re.search(self.patterns['legendary_actions_section'], text, re.IGNORECASE | re.DOTALL)
        if not legendary_match:
            return []

        legendary_text = legendary_match.group(1)
        actions = []

        # Split into individual actions
        action_parts = re.split(r'\n(?=[A-Z])', legendary_text.strip())
        for part in action_parts:
            if part.strip():
                actions.append(part.strip())

        return actions

    def _extract_description(self, text: str) -> str:
        """Extract descriptive text (simplified)"""
        lines = [line.strip() for line in text.split('\n') if line.strip()]

        # Look for descriptive paragraphs (not stat lines)
        description_lines = []
        for line in lines[1:6]:  # Check first few lines after name
            if (not re.match(r'(Tiny|Small|Medium|Large|Huge|Gargantuan)', line, re.IGNORECASE) and
                not re.match(r'Armor Class|Hit Points|Speed|STR|DEX', line, re.IGNORECASE) and
                len(line) > 20):
                description_lines.append(line)

        if description_lines:
            return ' '.join(description_lines[:2])

        return f"A {text.split()[0].lower() if text.split() else 'mysterious'} creature."

    def _infer_environment(self, name: str, text: str) -> List[str]:
        """Infer environment from name and description"""
        name_lower = name.lower()
        text_lower = text.lower()

        environments = []

        # Environment keywords
        env_keywords = {
            'Aquatic': ['water', 'ocean', 'sea', 'aquatic', 'swim'],
            'Underground': ['cave', 'underground', 'deep', 'dark'],
            'Forest': ['forest', 'tree', 'woodland'],
            'Mountain': ['mountain', 'peak', 'high altitude'],
            'Desert': ['desert', 'sand', 'arid'],
            'Swamp': ['swamp', 'marsh', 'bog'],
            'Urban': ['city', 'town', 'urban', 'civilized'],
            'Planar': ['plane', 'elemental', 'celestial', 'fiend', 'devil', 'demon']
        }

        for env, keywords in env_keywords.items():
            if any(keyword in name_lower or keyword in text_lower for keyword in keywords):
                environments.append(env)

        return environments if environments else ["Any"]

    def _standardize_ability_name(self, ability: str) -> Optional[str]:
        """Convert ability abbreviations to full names"""
        ability_map = {
            'str': 'Strength',
            'dex': 'Dexterity',
            'con': 'Constitution',
            'int': 'Intelligence',
            'wis': 'Wisdom',
            'cha': 'Charisma'
        }
        return ability_map.get(ability.lower()[:3])

def test_improved_parser():
    """Test the improved parser with sample data"""
    parser = ImprovedMonsterParser()

    # Test with sample monster block
    sample_block = """ABOLETH
Large aberration, lawful evil
Armor Class 17 (Natural Armor)
Hit Points 135 (18d12 + 18)
Speed 10 ft., swim 40 ft.
STR DEX CON INT WIS CHA
21 (+5) 9 (-1) 15 (+2) 18 (+4) 15 (+2) 18 (+4)
Saving Throws Con +6, Int +8, Wis +6
Skills History +12, Perception +10
Senses darkvision 120 ft., passive Perception 20
Languages Deep Speech, telepathy 120 ft.
Challenge 10 (5,900 XP)
Amphibious. The aboleth can breathe air and water.
Mucous Cloud. While underwater, the aboleth is surrounded by transformative mucus.
Actions
Multiattack. The aboleth makes three tentacle attacks.
Tentacle. Melee Weapon Attack: +9 to hit, reach 10 ft., one target.
Hit: 12 (2d6 + 5) bludgeoning damage."""

    monster = parser.parse_monster_block(sample_block)
    if monster:
        print(f"Successfully parsed: {monster.name}")
        print(f"CR: {monster.challenge_rating}")
        print(f"AC: {monster.combat_stats.armor_class}")
        print(f"HP: {monster.combat_stats.hit_points}")
        print(f"Attacks: {len(monster.attacks)}")
    else:
        print("Failed to parse sample monster")

# Run test if executed directly
if __name__ == "__main__":
    test_improved_parser()

Successfully parsed: Aboleth
CR: 10
AC: 17
HP: 135
Attacks: 0


In [2]:
from typing import Dict, List, Optional, Set, Any
from dataclasses import dataclass, field
import chromadb

@dataclass
class MonsterChunk:
    """A chunk of monster content for RAG retrieval"""
    monster_name: str
    content: str
    chunk_type: str  # 'stats', 'combat', 'abilities', 'description', 'tactics'
    metadata: Dict[str, Any]
    tags: Set[str] = field(default_factory=set)

    def get_retrieval_text(self) -> str:
        """Get formatted text for embedding"""
        return f"**{self.monster_name}** ({self.chunk_type})\n{self.content}"

class MonsterChunkCreator:
    """Create optimized chunks for monster RAG retrieval"""

    def __init__(self, max_tokens: int = 400):
        self.max_tokens = max_tokens

    def create_monster_chunks(self, monster) -> List[MonsterChunk]:
        """Create multiple chunks for a single monster"""
        chunks = []

        # Convert Monster to dict for metadata
        monster_dict = self._monster_to_dict(monster)

        # 1. Core Stats Chunk
        stats_chunk = self._create_stats_chunk(monster, monster_dict)
        chunks.append(stats_chunk)

        # 2. Combat Abilities Chunk
        combat_chunk = self._create_combat_chunk(monster, monster_dict)
        chunks.append(combat_chunk)

        # 3. Special Abilities Chunk (only if they exist)
        if monster.special_abilities:
            abilities_chunk = self._create_abilities_chunk(monster, monster_dict)
            chunks.append(abilities_chunk)

        # 4. Description & Environment Chunk
        desc_chunk = self._create_description_chunk(monster, monster_dict)
        chunks.append(desc_chunk)

        # 5. Additional features if they exist
        if monster.traits:
            traits_chunk = self._create_traits_chunk(monster, monster_dict)
            chunks.append(traits_chunk)

        if monster.legendary_actions:
            legendary_chunk = self._create_legendary_chunk(monster, monster_dict)
            chunks.append(legendary_chunk)

        return chunks

    def _monster_to_dict(self, monster) -> Dict[str, Any]:
        """Convert Monster to dictionary for metadata"""
        return {
            "name": monster.name,
            "size": monster.size,
            "creature_type": monster.creature_type,
            "alignment": monster.alignment,
            "challenge_rating": monster.challenge_rating,
            "environment": ', '.join(monster.environment) if monster.environment else "Any",
            "senses": monster.senses or "Unknown",
            "languages": monster.languages or "Unknown",
            "damage_resistances": ', '.join(monster.damage_resistances) if monster.damage_resistances else "None",
            "damage_immunities": ', '.join(monster.damage_immunities) if monster.damage_immunities else "None",
            "condition_immunities": ', '.join(monster.condition_immunities) if monster.condition_immunities else "None"
        }

    def _create_stats_chunk(self, monster, monster_dict: dict) -> MonsterChunk:
        """Create core statistics chunk"""
        content_parts = [
            f"**{monster.name}**",
            f"{monster.size} {monster.creature_type}, {monster.alignment}",
            f"**Challenge Rating:** {monster.challenge_rating}",
            f"**Armor Class:** {monster.combat_stats.armor_class}",
            f"**Hit Points:** {monster.combat_stats.hit_points} ({monster.combat_stats.hit_dice})",
            f"**Speed:** {', '.join([f'{k}: {v}' for k, v in monster.combat_stats.speed.items()])}",
        ]

        # Add ability scores
        abilities = monster.combat_stats.ability_scores
        content_parts.append(
            f"**Abilities:** STR {abilities.strength}, DEX {abilities.dexterity}, CON {abilities.constitution}, "
            f"INT {abilities.intelligence}, WIS {abilities.wisdom}, CHA {abilities.charisma}"
        )

        # Add saving throws if available
        if monster.combat_stats.saving_throws:
            saves = ", ".join([f"{k} {v:+d}" for k, v in monster.combat_stats.saving_throws.items()])
            content_parts.append(f"**Saving Throws:** {saves}")

        # Add skills if available
        if monster.combat_stats.skills:
            skills = ", ".join([f"{k} {v:+d}" for k, v in monster.combat_stats.skills.items()])
            content_parts.append(f"**Skills:** {skills}")

        # Add senses and languages
        if monster.senses:
            content_parts.append(f"**Senses:** {monster.senses}")
        if monster.languages:
            content_parts.append(f"**Languages:** {monster.languages}")

        content = "\n".join(content_parts)

        # Generate tags
        tags = self._generate_monster_tags(monster)
        tags.add("stats")
        tags.add("core")

        return MonsterChunk(
            monster_name=monster.name,
            content=content,
            chunk_type="stats",
            metadata=monster_dict,
            tags=tags
        )

    def _create_combat_chunk(self, monster, monster_dict: dict) -> MonsterChunk:
        """Create combat-focused chunk"""
        content_parts = [
            f"**{monster.name} - Combat**",
            f"CR {monster.challenge_rating}, AC {monster.combat_stats.armor_class}, HP {monster.combat_stats.hit_points}"
        ]

        # Add attacks
        if monster.attacks:
            content_parts.append("**Attacks:**")
            for attack in monster.attacks:
                attack_desc = f"- **{attack.name}:** +{attack.attack_bonus} to hit, {attack.damage_dice} {attack.damage_type}"
                if attack.range:
                    attack_desc += f" ({attack.range})"
                if attack.description:
                    attack_desc += f". {attack.description}"
                content_parts.append(attack_desc)

        # Add damage resistances/immunities
        if monster.damage_resistances:
            content_parts.append(f"**Damage Resistances:** {', '.join(monster.damage_resistances)}")
        if monster.damage_immunities:
            content_parts.append(f"**Damage Immunities:** {', '.join(monster.damage_immunities)}")
        if monster.condition_immunities:
            content_parts.append(f"**Condition Immunities:** {', '.join(monster.condition_immunities)}")

        content = "\n".join(content_parts)

        tags = self._generate_monster_tags(monster)
        tags.update(["combat", "attacks", "damage"])

        return MonsterChunk(
            monster_name=monster.name,
            content=content,
            chunk_type="combat",
            metadata=monster_dict,
            tags=tags
        )

    def _create_abilities_chunk(self, monster, monster_dict: dict) -> MonsterChunk:
        """Create special abilities chunk"""
        content_parts = [
            f"**{monster.name} - Special Abilities**",
            f"CR {monster.challenge_rating}"
        ]

        if monster.special_abilities:
            content_parts.append("**Special Abilities:**")
            for ability in monster.special_abilities:
                content_parts.append(f"- {ability}")

        content = "\n".join(content_parts)

        tags = self._generate_monster_tags(monster)
        tags.add("abilities")
        tags.add("special")

        return MonsterChunk(
            monster_name=monster.name,
            content=content,
            chunk_type="abilities",
            metadata=monster_dict,
            tags=tags
        )

    def _create_description_chunk(self, monster, monster_dict: dict) -> MonsterChunk:
        """Create description and environment chunk"""
        content_parts = [
            f"**{monster.name} - Description & Environment**",
            f"{monster.size} {monster.creature_type}, {monster.alignment}"
        ]

        if monster.description:
            # Truncate if too long
            desc = monster.description
            if len(desc) > 300:
                desc = desc[:300] + "..."
            content_parts.append(f"**Description:** {desc}")

        if monster.environment:
            content_parts.append(f"**Environment:** {', '.join(monster.environment)}")

        content = "\n".join(content_parts)

        tags = self._generate_monster_tags(monster)
        tags.update(["description", "environment", "lore"])

        return MonsterChunk(
            monster_name=monster.name,
            content=content,
            chunk_type="description",
            metadata=monster_dict,
            tags=tags
        )

    def _create_traits_chunk(self, monster, monster_dict: dict) -> MonsterChunk:
        """Create traits chunk"""
        content_parts = [
            f"**{monster.name} - Traits**",
            f"CR {monster.challenge_rating}"
        ]

        if monster.traits:
            content_parts.append("**Traits:**")
            for trait in monster.traits:
                content_parts.append(f"- {trait}")

        content = "\n".join(content_parts)

        tags = self._generate_monster_tags(monster)
        tags.add("traits")

        return MonsterChunk(
            monster_name=monster.name,
            content=content,
            chunk_type="traits",
            metadata=monster_dict,
            tags=tags
        )

    def _create_legendary_chunk(self, monster, monster_dict: dict) -> MonsterChunk:
        """Create legendary actions chunk"""
        content_parts = [
            f"**{monster.name} - Legendary Actions**",
            f"CR {monster.challenge_rating}"
        ]

        if monster.legendary_actions:
            content_parts.append("**Legendary Actions:**")
            for action in monster.legendary_actions:
                content_parts.append(f"- {action}")

        content = "\n".join(content_parts)

        tags = self._generate_monster_tags(monster)
        tags.add("legendary")
        tags.add("actions")

        return MonsterChunk(
            monster_name=monster.name,
            content=content,
            chunk_type="legendary",
            metadata=monster_dict,
            tags=tags
        )

    def _generate_monster_tags(self, monster) -> Set[str]:
        """Generate tags for retrieval optimization"""
        tags = set()

        # Basic tags
        cr_clean = str(monster.challenge_rating).replace('/', '_')
        tags.add(f"cr_{cr_clean}")
        tags.add(f"size_{monster.size.lower()}")
        tags.add(f"type_{monster.creature_type.lower().replace(' ', '_')}")

        # Alignment tags
        alignment = monster.alignment.lower()
        if 'lawful' in alignment:
            tags.add("alignment_lawful")
        if 'chaotic' in alignment:
            tags.add("alignment_chaotic")
        if 'good' in alignment:
            tags.add("alignment_good")
        if 'evil' in alignment:
            tags.add("alignment_evil")
        if 'neutral' in alignment:
            tags.add("alignment_neutral")

        # Environment tags
        if monster.environment:
            for env in monster.environment:
                env_clean = env.lower().replace(" ", "_").replace(",", "")
                tags.add(f"env_{env_clean}")

        # Combat role tags (infer from stats)
        abilities = monster.combat_stats.ability_scores
        if abilities.strength >= 18:
            tags.add("role_melee")
            tags.add("strength_based")
        if abilities.dexterity >= 16:
            tags.add("role_ranged")
            tags.add("dexterity_based")
        if abilities.intelligence >= 16 or abilities.charisma >= 16:
            tags.add("role_caster")
            tags.add("spellcaster")
        if abilities.constitution >= 16:
            tags.add("tank")

        # Damage type tags
        if monster.attacks:
            for attack in monster.attacks:
                dmg_type = attack.damage_type.lower().replace(" ", "_")
                tags.add(f"damage_{dmg_type}")

        return tags

In [3]:
import chromadb
from typing import List, Dict, Optional, Any
import uuid

class MonsterChromaManager:
    """ChromaDB manager for monster collections"""

    def __init__(self, db_path: str = "./chromadb"):
        self.db_path = db_path
        self.client = chromadb.PersistentClient(path=db_path)

        # Create or get monster collection
        self.collection = self.client.get_or_create_collection(
            name="monsters",
            metadata={"description": "D&D 5e Monster Manual creatures for RAG system"}
        )

        print(f"ChromaDB Monster Manager initialized:")
        print(f"  Database path: {db_path}")
        print(f"  Collection: {self.collection.name}")
        print(f"  Current documents: {self.collection.count()}")

    def add_monster_chunks(self, chunks: List[MonsterChunk]):
        """Add monster chunks to database with ChromaDB-compatible metadata"""
        if not chunks:
            print("No chunks to add")
            return

        documents = []
        metadatas = []
        ids = []

        for i, chunk in enumerate(chunks):
            # Create unique ID
            chunk_id = f"{chunk.monster_name.lower().replace(' ', '_')}_{chunk.chunk_type}_{i}"
            ids.append(chunk_id)

            # Get retrieval text
            documents.append(chunk.get_retrieval_text())

            # Convert metadata to ChromaDB-compatible format (all values must be strings/numbers)
            metadata = {}
            for key, value in chunk.metadata.items():
                if isinstance(value, list):
                    metadata[key] = ", ".join(str(item) for item in value) if value else "None"
                elif value is None:
                    metadata[key] = "Unknown"
                else:
                    metadata[key] = str(value)

            # Add chunk-specific metadata
            metadata["chunk_type"] = chunk.chunk_type
            metadata["monster_name"] = chunk.monster_name
            metadata["tags"] = ", ".join(sorted(chunk.tags)) if chunk.tags else ""

            metadatas.append(metadata)

        try:
            self.collection.add(
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )
            print(f"Successfully added {len(chunks)} chunks to ChromaDB")
        except Exception as e:
            print(f"Error adding chunks to ChromaDB: {e}")

    def add_monsters(self, monsters: List, chunk_creator: MonsterChunkCreator):
        """Add multiple monsters to database by creating chunks for each"""
        total_chunks = 0

        for monster in monsters:
            try:
                chunks = chunk_creator.create_monster_chunks(monster)
                self.add_monster_chunks(chunks)
                total_chunks += len(chunks)
                print(f"Added {monster.name}: {len(chunks)} chunks")
            except Exception as e:
                print(f"Error processing monster {getattr(monster, 'name', 'Unknown')}: {e}")
                continue

        print(f"Total chunks added: {total_chunks}")
        return total_chunks

    def search_monsters(self, query: str, n_results: int = 5, filters: Optional[Dict] = None):
        """Search monsters with optional filters"""
        try:
            if filters:
                results = self.collection.query(
                    query_texts=[query],
                    n_results=n_results,
                    where=filters
                )
            else:
                results = self.collection.query(
                    query_texts=[query],
                    n_results=n_results
                )
            return results
        except Exception as e:
            print(f"Search error: {e}")
            return {"documents": [[]], "metadatas": [[]], "ids": [[]]}

    def search_by_challenge_rating(self, cr: str, n_results: int = 10):
        """Search monsters by challenge rating"""
        filters = {"challenge_rating": cr}
        return self.search_monsters(f"challenge rating {cr}", n_results, filters)

    def search_by_type(self, creature_type: str, n_results: int = 10):
        """Search monsters by creature type"""
        filters = {"creature_type": creature_type}
        return self.search_monsters(creature_type, n_results, filters)

    def search_by_environment(self, environment: str, n_results: int = 10):
        """Search monsters by environment"""
        return self.search_monsters(f"environment {environment}", n_results)

    def get_monster_names(self) -> List[str]:
        """Get list of all monster names in database"""
        try:
            # Get all documents
            all_results = self.collection.get()
            monster_names = set()

            if all_results["metadatas"]:
                for metadata in all_results["metadatas"]:
                    if "monster_name" in metadata:
                        monster_names.add(metadata["monster_name"])

            return sorted(list(monster_names))
        except Exception as e:
            print(f"Error getting monster names: {e}")
            return []

    def get_collection_stats(self):
        """Get collection statistics"""
        try:
            total_docs = self.collection.count()
            monster_names = self.get_monster_names()

            # Get chunk type distribution
            all_results = self.collection.get()
            chunk_types = {}

            if all_results["metadatas"]:
                for metadata in all_results["metadatas"]:
                    chunk_type = metadata.get("chunk_type", "unknown")
                    chunk_types[chunk_type] = chunk_types.get(chunk_type, 0) + 1

            return {
                "total_documents": total_docs,
                "unique_monsters": len(monster_names),
                "collection_name": self.collection.name,
                "chunk_types": chunk_types,
                "sample_monsters": monster_names[:10] if monster_names else []
            }
        except Exception as e:
            print(f"Error getting stats: {e}")
            return {"error": str(e)}

    def clear_collection(self):
        """Clear all data from collection (use with caution!)"""
        try:
            # Delete the collection and recreate it
            self.client.delete_collection("monsters")
            self.collection = self.client.create_collection(
                name="monsters",
                metadata={"description": "D&D 5e Monster Manual creatures for RAG system"}
            )
            print("Collection cleared successfully")
        except Exception as e:
            print(f"Error clearing collection: {e}")

    def preview_search_results(self, query: str, n_results: int = 3):
        """Preview search results in a readable format"""
        results = self.search_monsters(query, n_results)

        if not results["documents"] or not results["documents"][0]:
            print(f"No results found for '{query}'")
            return

        print(f"Search Results for '{query}':")
        print("=" * 50)

        for i, (doc, meta, doc_id) in enumerate(zip(
            results["documents"][0],
            results["metadatas"][0],
            results["ids"][0]
        )):
            print(f"\nResult {i+1}:")
            print(f"  ID: {doc_id}")
            print(f"  Monster: {meta.get('monster_name', 'Unknown')}")
            print(f"  Type: {meta.get('chunk_type', 'Unknown')}")
            print(f"  CR: {meta.get('challenge_rating', 'Unknown')}")
            print(f"  Content: {doc[:200]}...")

        print("\n" + "=" * 50)

In [4]:
# Complete pipeline to process Monster Manual PDF and add to ChromaDB
class MonsterPDFToChromaPipeline:
    """Complete pipeline: PDF -> Parse -> Chunk -> ChromaDB"""

    def __init__(self, db_path: str = "./chromadb"):
        self.pdf_parser = ImprovedMonsterParser()
        self.chunk_creator = MonsterChunkCreator()
        self.chroma_manager = MonsterChromaManager(db_path)

    def process_pdf_to_chromadb(self, pdf_path: str, clear_existing: bool = False) -> int:
        """Process PDF and add monsters to ChromaDB"""
        print("Starting Monster PDF to ChromaDB pipeline...")

        # Clear existing data if requested
        if clear_existing:
            print("Clearing existing collection...")
            self.chroma_manager.clear_collection()

        # Step 1: Extract PDF text
        print("\n1. Extracting PDF text...")
        text = self.pdf_parser.extract_pdf_text(pdf_path)
        if not text:
            print("Failed to extract PDF text")
            return 0

        # Step 2: Split into monster blocks
        print("\n2. Splitting into monster blocks...")
        monster_blocks = self.pdf_parser.split_into_monster_blocks(text)

        # Step 3: Parse each block
        print("\n3. Parsing monster blocks...")
        monsters = []
        successful_parses = 0
        failed_parses = 0

        for i, block in enumerate(monster_blocks):
            if i % 20 == 0:
                print(f"   Processing block {i+1}/{len(monster_blocks)}")

            monster = self.pdf_parser.parse_monster_block(block)
            if monster:
                monsters.append(monster)
                successful_parses += 1
                if successful_parses % 25 == 0:
                    print(f"   Successfully parsed {successful_parses} monsters...")
            else:
                failed_parses += 1

        print(f"Parsing complete: {successful_parses} success, {failed_parses} failed")

        # Step 4: Create chunks and add to ChromaDB
        print("\n4. Creating chunks and adding to ChromaDB...")
        total_chunks = self.chroma_manager.add_monsters(monsters, self.chunk_creator)

        print(f"\nPipeline complete!")
        print(f"  Parsed {successful_parses} monsters")
        print(f"  Created {total_chunks} chunks")
        print(f"  Added to ChromaDB collection")

        return total_chunks

    def test_search(self, queries: List[str] = None):
        """Test the search functionality with sample queries"""
        if queries is None:
            queries = [
                "dragon fire breath",
                "goblin stealth attack",
                "undead necrotic damage",
                "challenge rating 10",
                "legendary actions"
            ]

        print("\nTesting search functionality:")
        print("=" * 40)

        for query in queries:
            print(f"\nQuery: '{query}'")
            print("-" * 30)
            self.chroma_manager.preview_search_results(query, n_results=2)

    def show_statistics(self):
        """Display collection statistics"""
        print("\nCollection Statistics:")
        print("=" * 30)

        stats = self.chroma_manager.get_collection_stats()

        for key, value in stats.items():
            if key == "chunk_types":
                print(f"{key}:")
                for chunk_type, count in value.items():
                    print(f"  {chunk_type}: {count}")
            elif key == "sample_monsters":
                print(f"{key}: {', '.join(value[:5])}{'...' if len(value) > 5 else ''}")
            else:
                print(f"{key}: {value}")

# Usage example for Jupyter notebook
def run_monster_pipeline(pdf_path: str, test_search: bool = True):
    """Complete workflow to process Monster Manual"""

    # Initialize pipeline
    pipeline = MonsterPDFToChromaPipeline()

    # Process the PDF
    total_chunks = pipeline.process_pdf_to_chromadb(pdf_path, clear_existing=True)

    if total_chunks > 0:
        # Show statistics
        pipeline.show_statistics()

        # Test search if requested
        if test_search:
            pipeline.test_search()

        print(f"\nSuccess! {total_chunks} chunks added to ChromaDB")
    else:
        print("Pipeline failed - no chunks were created")

    return pipeline

# Quick test function
def quick_test_search(db_path: str = "./chromadb"):
    """Quick function to test search on existing database"""
    manager = MonsterChromaManager(db_path)

    # Show stats
    stats = manager.get_collection_stats()
    print(f"Database contains {stats['total_documents']} documents from {stats['unique_monsters']} monsters")

    # Test searches
    test_queries = [
        "aboleth tentacle attack",
        "dragon legendary actions",
        "goblin small humanoid",
        "challenge rating 5"
    ]

    for query in test_queries:
        print(f"\nSearching: '{query}'")
        manager.preview_search_results(query, n_results=2)

In [5]:
# Process your Monster Manual PDF
pdf_path = "Dungeons and Dragons - Monster Manual (Skip Williams, Jonathan Tweet, Monte Cook) (Z-Library).pdf"
pipeline = run_monster_pipeline(pdf_path, test_search=True)

ChromaDB Monster Manager initialized:
  Database path: ./chromadb
  Collection: monsters
  Current documents: 1488
Starting Monster PDF to ChromaDB pipeline...
Clearing existing collection...
Collection cleared successfully

1. Extracting PDF text...
Extracting text from Dungeons and Dragons - Monster Manual (Skip Williams, Jonathan Tweet, Monte Cook) (Z-Library).pdf...
Processing 334 pages...
Processed 50/334 pages...
Processed 100/334 pages...
Processed 150/334 pages...
Processed 200/334 pages...
Processed 250/334 pages...
Processed 300/334 pages...
Extraction complete! 1,664,059 characters extracted

2. Splitting into monster blocks...
Found Aboleth at line 444
Split into 400 monster blocks

3. Parsing monster blocks...
   Processing block 1/400
   Processing block 21/400
   Processing block 41/400
   Processing block 61/400
   Processing block 81/400
   Successfully parsed 25 monsters...
   Processing block 101/400
   Processing block 121/400
   Processing block 141/400
   Processi

In [6]:
# Test specific searches
pipeline.chroma_manager.preview_search_results("goblin", n_results=3)
pipeline.chroma_manager.preview_search_results("dragon fire", n_results=3)

Search Results for 'goblin':

Result 1:
  ID: hobgoblin_stats_0
  Monster: Hobgoblin
  Type: stats
  CR: Rating:
  Content: **Hobgoblin** (stats)
**Hobgoblin**
Medium Humanoid, Goblinoid
**Challenge Rating:** Rating:
**Armor Class:** 10
**Hit Points:** 1 (1d8)
**Speed:** walk: 30 ft
**Abilities:** STR 10, DEX 10, CON 10, I...

Result 2:
  ID: hobgoblin_description_3
  Monster: Hobgoblin
  Type: description
  CR: Rating:
  Content: **Hobgoblin** (description)
**Hobgoblin - Description & Environment**
Medium Humanoid, Goblinoid
**Description:** Hobgoblin, 1st-Level Warrior Hit Dice: 1d8+2 (6 hp)
**Environment:** Underground, Moun...

Result 3:
  ID: bugbear_stats_0
  Monster: Bugbear
  Type: stats
  CR: Rating:
  Content: **Bugbear** (stats)
**Bugbear**
Medium Humanoid, Goblinoid Bugbears survive primarily by hunting and they eat whatever
**Challenge Rating:** Rating:
**Armor Class:** 10
**Hit Points:** 1 (1d8)
**Speed...

Search Results for 'dragon fire':

Result 1:
  ID: agon_descripti

In [8]:
# Search by challenge rating
results = pipeline.chroma_manager.search_by_challenge_rating("10", n_results=5)

# Search by creature type
results = pipeline.chroma_manager.search_by_type("dragon", n_results=5)

# Search by environment
results = pipeline.chroma_manager.search_by_environment("underground", n_results=5)